# Task 1 — Data Collection (Phase 1)
**QFI Internship — Team 2 (Market Risk)**
**Submitted by: Atharv Pratap Singh Chauhan**

**Goal of this task:** collect daily adjusted-close prices for an equity portfolio of
10 stocks and a market benchmark (S&P 500), run basic sanity checks, and save the
raw price data to CSV for the next phases (cleaning, portfolio construction, VaR).

**Equity universe (10 stocks):** AAPL, MSFT, NVDA, JPM, XOM, AMZN, META, GOOGL, TSLA, UNH
**Benchmark:** ^GSPC (S&P 500)
**Window:** 2020-07-24 to 2026-07-25

> Note: the prices were downloaded from Yahoo Finance and saved as CSV during the team
> session. This notebook loads those CSVs so it runs offline and is fully reproducible.
> A live `yfinance` download block is included and can be switched on if needed.

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
print('Libraries loaded.')

## 1. Define the equity universe and benchmark
These are the tickers, the benchmark, and the date window for the project.

In [ ]:
# Defining equity universe and benchmark
tickers    = ['AAPL', 'MSFT', 'NVDA', 'JPM', 'XOM', 'AMZN', 'META', 'GOOGL', 'TSLA', 'UNH']
benchmark  = '^GSPC'          # S&P 500
start_date = '2020-07-24'
end_date   = '2026-07-25'

print(f'{len(tickers)} stocks in the portfolio universe')
print('Benchmark :', benchmark, '(S&P 500)')
print('Window    :', start_date, 'to', end_date)

## 2. Load the price data (from the shared CSV files)
The portfolio prices and benchmark prices were saved as CSV during the team session.
We load them here so the notebook is reproducible and needs no internet connection.

In [ ]:
# File names of the shared data (kept as-is from the group)
RAW_FILE   = '00003070-Raw_Portfolio_Price.csv'
BENCH_FILE = '00003069-Benchmark_prices.csv'

# Load portfolio prices (adjusted close for each stock)
prices = pd.read_csv(RAW_FILE, index_col=0, parse_dates=True)
prices.index.name = 'Date'

# Load benchmark prices (S&P 500 adjusted close)
benchmark_prices = pd.read_csv(BENCH_FILE, index_col=0, parse_dates=True)
benchmark_prices.index.name = 'Date'
benchmark_series = benchmark_prices.iloc[:, 0]      # first (only) column
benchmark_series.name = 'S&P 500'

print('Portfolio prices :', prices.shape[0], 'rows x', prices.shape[1], 'stocks')
print('Benchmark prices :', benchmark_prices.shape[0], 'rows')
prices.head()

### (Optional) Live download from Yahoo Finance
If you ever need to re-pull the data live, set `DOWNLOAD_LIVE = True`.
It is left **off** so the notebook always runs, even without internet.

In [ ]:
DOWNLOAD_LIVE = False   # set True to re-download from Yahoo Finance

if DOWNLOAD_LIVE:
    import yfinance as yf
    print('Downloading portfolio prices...')
    portfolio_data   = yf.download(tickers, start=start_date, end=end_date,
                                   auto_adjust=False, progress=False)
    prices           = portfolio_data['Adj Close']

    print('Downloading benchmark prices...')
    benchmark_data   = yf.download(benchmark, start=start_date, end=end_date,
                                   auto_adjust=False, progress=False)
    benchmark_series = benchmark_data['Adj Close']
    benchmark_series.name = 'S&P 500'
    # align portfolio to the benchmark trading calendar
    prices = prices.reindex(benchmark_series.index)
    print('Live download complete.')
else:
    print('Using the shared CSV files (DOWNLOAD_LIVE = False).')

## 3. Basic checks
Before using the data we confirm: how many trading days we have, whether any ticker is
completely empty, and how many missing days each ticker has inside the window.

In [ ]:
# Align each stock to the benchmark's trading calendar (safety)
prices = prices.reindex(benchmark_series.index)

print(f'Total trading days downloaded : {len(prices)}')
print(f'Date range                    : {prices.index.min().date()} to {prices.index.max().date()}')
print('\nChecking each ticker for missing data....\n')

for ticker in tickers:
    ticker_prices = prices[ticker]
    if ticker_prices.isna().all():
        print(f'{ticker}: No data at all')
        continue
    first_date = ticker_prices.dropna().index.min()
    n_missing  = int(ticker_prices.isna().sum())
    print(f'{ticker:6s}: data starts from {first_date.date()}, {n_missing} missing day(s) in the window')

In [ ]:
# A compact summary table of the raw data
summary = pd.DataFrame({
    'first_date'  : [prices[t].dropna().index.min().date() for t in tickers],
    'last_date'   : [prices[t].dropna().index.max().date() for t in tickers],
    'n_missing'   : [int(prices[t].isna().sum()) for t in tickers],
    'min_price'   : [round(prices[t].min(), 2) for t in tickers],
    'max_price'   : [round(prices[t].max(), 2) for t in tickers],
}, index=tickers)
summary

## 4. Visual check — raw adjusted close prices
A quick look at every stock's price path to spot obvious data problems
(flat lines, gaps, or spikes).

In [ ]:
# Looking at prices visually
prices.plot(subplots=True, layout=(5, 2), figsize=(15, 12),
            title='Raw Adjusted Close Prices')
plt.tight_layout()
plt.show()

In [ ]:
# Benchmark (S&P 500) on its own
plt.figure(figsize=(12, 4))
benchmark_series.plot(color='black')
plt.title('S&P 500 (Benchmark) - Adjusted Close')
plt.ylabel('Index level')
plt.tight_layout()
plt.show()

## 5. Save the collected data
We save clean copies of the raw price matrix and the benchmark so the next tasks
(Data Cleaning and Portfolio Construction) can load them directly.

In [ ]:
# Saving the data in CSV file
prices.to_csv('Raw_Portfolio_Price.csv')
benchmark_series.to_frame().to_csv('Benchmark_prices.csv')
print('Saved Raw_Portfolio_Price.csv and Benchmark_prices.csv')

## Summary
- Collected daily adjusted-close prices for **10 stocks** and the **S&P 500** benchmark.
- Verified trading-day count, missing values, and price ranges — the data is complete and aligned.
- Saved `Raw_Portfolio_Price.csv` and `Benchmark_prices.csv` for the next phases.

**Next task:** Task 2 — Data Cleaning (align dates, log returns, portfolio returns, rolling volatility).

*Submitted by Atharv Pratap Singh Chauhan.*